# Star Schema

In [447]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import sqlite3
import matplotlib.pyplot as plt
import duckdb
from IPython.display import Image, display

### Caminho base do projeto

In [448]:
BASE_PATH = Path().resolve()

while BASE_PATH.name != "lh-nautical-data-project":
    BASE_PATH = BASE_PATH.parent

print(f"BASE PATH: {BASE_PATH}")

BASE PATH: /Users/richardgomes/lh-nautical-data-project


Caminhos para as pastas do projeto.

In [449]:
DATA_PATH = BASE_PATH / "data"

RAW_PATH = DATA_PATH / "raw"
STAGING_PATH = DATA_PATH / "staging"
INTERMEDIATE_PATH = DATA_PATH / "intermediate"
MARTS_PATH = DATA_PATH / "marts"

SQL_PATH = BASE_PATH / "sql"
IMAGES_PATH = BASE_PATH / "imagens"

Carregamento das tabelas

In [450]:
df_vendas = pd.read_csv(STAGING_PATH / "stg_vendas.csv")
df_clientes = pd.read_csv(STAGING_PATH / "stg_clientes.csv")
df_produtos = pd.read_csv(STAGING_PATH / "stg_produtos.csv")
df_custos = pd.read_csv(STAGING_PATH / "stg_custos_importacao_tratado.csv")
df_cambio = pd.read_csv(STAGING_PATH / "cambio_diario.csv")
df_indicadores = pd.read_csv(STAGING_PATH / "indicadores_financeiros_produtos.csv")

Visualização das colunas:

In [451]:
df_vendas.head()




,id_venda,id_cliente,id_produto,quantidade,valor_total,data_venda
0,1230,17,91,4,512566.80,2023-01-01
1,2300,30,95,9,596858.40,2023-01-01
2,3131,28,130,13,53873.00,2023-01-01
3,4212,9,96,6,402538.75,2023-01-01
4,4294,7,44,5,51332.30,2023-01-01


In [452]:
df_clientes.head()

,id_cliente,nome_cliente,email,cidade,estado
0,1,Femininos Oliveira Antunes,femininos.oliveira.antunes@icloud.com,Aratu,BA
1,2,Fernanda Azevedo Soares Nunes Vieira,nunes.fernanda.soares.azevedo.vieira@outlook.com,Recife,PE
2,3,Daniel Farias Ribeiro Teixeira,farias.teixeira.daniel.ribeiro@gmail.com,Rio Grande,RS
3,4,Thiago Moreira,thiago.moreira@gmail.com,Rio Branco,AC
4,5,Pedro Freitas,pedro.freitas@icloud.com,Santarém Novo,PA


In [453]:
df_produtos.head()




,nome_produto,preco_base_produto,id_produto,categoria_produto
0,Transponder AIS Maré Magnum,33122.52,1,eletronicos
1,Transponder Furuno Marlin,13998.15,2,eletronicos
2,Radar Furuno Pulse Leviathan,9024.19,3,eletronicos
3,Rádio AIS Hydro Tidal Zen,3381.88,4,eletronicos
4,Piloto Automático Furuno Storm,23669.01,5,eletronicos


In [454]:
df_custos.head()

,id_produto,nome_produto,categoria,data_inicio,preco_usd
0,1,Transponder Ais Maré Magnum,eletrônicos,2016-08-10,10583.63
1,1,Transponder Ais Maré Magnum,eletrônicos,2018-06-15,8778.36
2,1,Transponder Ais Maré Magnum,eletrônicos,2018-09-25,8023.87
3,1,Transponder Ais Maré Magnum,eletrônicos,2019-03-19,8772.78
4,1,Transponder Ais Maré Magnum,eletrônicos,2020-01-17,7918.18


In [455]:
df_cambio.head()

,data,taxa_cambio
0,2023-01-02,5.3436
1,2023-01-03,5.3759
2,2023-01-04,5.4459
3,2023-01-05,5.4026
4,2023-01-06,5.2855


In [456]:
df_indicadores.head()

,id_produto,receita_total,prejuizo_total,percentual_prejuizo
0,1,17187280.35,154.709200,0.000009
1,2,8862231.15,21.358768,0.000002
2,3,4322136.25,2318.092263,0.000536
3,4,1915160.00,29.978055,0.000016
4,5,13494880.35,1091.034870,0.000081


### Criação camada intermediate

Para esse desafio vou utilizar apenas uma camada intermediate para enriquecer a tabela de vendas. (int_vendas_enriquecida)

É o ponto com regra de negócio mais complexa. 

Vou usar as dimensões geradas direto da camada staging pois já estão tratadas e normalizadas.

Sei que não é o padrão no dia a dia de um Analytics Engineer, mas meu tempo para esse desafio é curto!

Ajuste nas datas.

In [457]:
df_vendas["data_venda"] = pd.to_datetime(df_vendas["data_venda"])
df_custos["data_inicio"] = pd.to_datetime(df_custos["data_inicio"])
df_cambio["data"] = pd.to_datetime(df_cambio["data"])

Join vendas + custos (histórico)

In [458]:
df_merge = df_vendas.merge(df_custos, on="id_produto", how="left")
df_merge = df_merge[df_merge["data_inicio"] <= df_merge["data_venda"]]
df_merge = df_merge.sort_values(
    ["id_venda", "data_inicio"],
    ascending=[True, False]
)

df_merge = df_merge.drop_duplicates(subset=["id_venda"])


Join com o câmbio

In [459]:
df_merge = df_merge.sort_values("data_venda")
df_cambio = df_cambio.sort_values("data")


df_merge = pd.merge_asof(
    df_merge,
    df_cambio,
    left_on="data_venda",
    right_on="data",
    direction="forward" 
)

Cálculos

In [460]:
df_merge["custo_unitario_brl"] = df_merge["preco_usd"] * df_merge["taxa_cambio"]
df_merge["custo_total_brl"] = df_merge["custo_unitario_brl"] * df_merge["quantidade"]

Prejuízo só quando custo > venda

In [461]:
df_merge["prejuizo"] = (df_merge["custo_total_brl"] - df_merge["valor_total"]).clip(lower=0)

In [462]:
int_vendas_enriquecida = df_merge[[
    "id_venda",
    "id_cliente",
    "id_produto",
    "data_venda",
    "quantidade",
    "valor_total",
    "preco_usd",
    "taxa_cambio",
    "custo_unitario_brl",
    "custo_total_brl",
    "prejuizo"
]]


Materializando tabela.

In [463]:
INTERMEDIATE_PATH = DATA_PATH / "intermediate"

int_vendas_enriquecida.to_csv(
    INTERMEDIATE_PATH / "int_vendas_enriquecida.csv",
    index=False
)

In [464]:
int_vendas_enriquecida.head()

,id_venda,id_cliente,id_produto,data_venda,quantidade,valor_total,preco_usd,taxa_cambio,custo_unitario_brl,custo_total_brl,prejuizo
0,1230,17,91,2023-01-01,4,512566.80,26303.31,5.3436,140554.367316,562217.469264,49650.669264
1,2300,30,95,2023-01-01,9,596858.40,12945.63,5.3436,69176.268468,622586.416212,25728.016212
2,3131,28,130,2023-01-01,13,53873.00,749.89,5.3436,4007.112204,52092.458652,0.000000
3,4212,9,96,2023-01-01,6,402538.75,13063.42,5.3436,69805.691112,418834.146672,16295.396672
4,4294,7,44,2023-01-01,5,51332.30,1963.02,5.3436,10489.593672,52447.968360,1115.668360


In [465]:
int_vendas_enriquecida.isnull().sum()

id_venda              0
id_cliente            0
id_produto            0
data_venda            0
quantidade            0
valor_total           0
preco_usd             0
taxa_cambio           0
custo_unitario_brl    0
custo_total_brl       0
prejuizo              0
dtype: int64

Para este desafio, foi utilizada uma única camada intermediate para enriquecer a tabela de vendas (int_vendas_enriquecida), concentrando as regras de negócio mais complexas.

As dimensões foram geradas diretamente a partir da camada staging, pois já estavam devidamente tratadas e normalizadas.

Essa abordagem permite separar claramente ingestão, transformação e consumo analítico, mantendo o pipeline simples e eficiente.

### Criação camada Marts

dim_cliente

In [466]:
dim_cliente = df_clientes.copy()
dim_cliente = dim_cliente.drop_duplicates(subset=["id_cliente"])


dim_cliente.to_csv(
    MARTS_PATH / "dim_cliente.csv",
    index=False
)

print("dim_cliente criada!")

dim_cliente criada!


dim_produto

In [467]:
dim_produto = df_produtos.copy()
dim_produto = dim_produto.drop_duplicates(subset=["id_produto"])

dim_produto = dim_produto[
    ["id_produto", "nome_produto", "categoria_produto", "preco_base_produto"]
]
dim_produto.to_csv(
    MARTS_PATH / "dim_produto.csv",
    index=False
)

print("dim_produto criada!")

dim_produto criada!


dim_date

Ajuste nas colunas da dim_date criada anteriormente durante as questões

In [468]:
dim_date = pd.read_csv(MARTS_PATH / "dim_date.csv")
dim_date["data"] = pd.to_datetime(dim_date["data"])


dim_date = dim_date.sort_values("data")


dim_date = dim_date[
    [
        "data",
        "ano",
        "mes",
        "dia",
        "dia_da_semana",
        "nome_mes",
        "dia_da_semana_num",
        "fim_de_semana",
        "data_chave"
    ]
]

In [469]:
dim_date.to_csv(
    MARTS_PATH / "dim_date.csv",
    index=False
)

print("dim_date ajustada!")

dim_date ajustada!


Tabela Fato -  fct_vendas

In [470]:
fct_vendas = int_vendas_enriquecida[
    [
        "id_venda",
        "id_cliente",
        "id_produto",
        "data_venda",
        "quantidade",
        "valor_total",
        "custo_total_brl",
        "prejuizo"
    ]
].copy()

In [471]:
fct_vendas = fct_vendas.drop_duplicates(subset=["id_venda"])

fct_vendas.to_csv(
    MARTS_PATH / "fct_vendas.csv",
    index=False
)

print("fct_vendas criada!")

fct_vendas criada!


In [472]:
fct_vendas.isnull().sum()

id_venda           0
id_cliente         0
id_produto         0
data_venda         0
quantidade         0
valor_total        0
custo_total_brl    0
prejuizo           0
dtype: int64

In [473]:
fct_vendas.head()


,id_venda,id_cliente,id_produto,data_venda,quantidade,valor_total,custo_total_brl,prejuizo
0,1230,17,91,2023-01-01,4,512566.80,562217.469264,49650.669264
1,2300,30,95,2023-01-01,9,596858.40,622586.416212,25728.016212
2,3131,28,130,2023-01-01,13,53873.00,52092.458652,0.000000
3,4212,9,96,2023-01-01,6,402538.75,418834.146672,16295.396672
4,4294,7,44,2023-01-01,5,51332.30,52447.968360,1115.668360
